# Arvore para detecção de fraude

## 1. Carregamento dos dados

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
import os

drive.mount('/content/drive')

caminho = "/content/drive/MyDrive/Aula_llm/Trabalho final/Dados/IBM/"

X_train_balanceado = pd.read_csv(caminho + 'dados_trans_treino.csv')
X_val_processado = pd.read_csv(caminho + 'dados_trans_validacao.csv')
X_test_processado = pd.read_csv(caminho + 'dados_trans_teste.csv')
X_accounts = pd.read_csv(caminho + 'LI-Small_accounts.csv')

print(X_train_balanceado.shape)
print(X_val_processado.shape)
print(X_test_processado.shape)
print(X_accounts.shape)

Mounted at /content/drive
(27450, 43)
(5882, 43)
(5883, 43)
(712688, 5)


In [2]:
print(X_train_balanceado.columns)
print(X_accounts.columns)

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering', 'ano', 'mes',
       'dia_mes', 'hora', 'minuto', 'dia_semana', 'semana_ano', 'semana_mes',
       'fim_de_semana', 'tipo_semana', 'periodo_dia', 'periodo_dia_3',
       'inicio_mes', 'fim_mes', 'hora_sin', 'hora_cos', 'dia_semana_sin',
       'dia_semana_cos', 'mes_sin', 'mes_cos', 'dif_valor', 'dif_valor_abs',
       'razao_pago_recebido', 'amount_paid_log', 'amount_received_log',
       'mesmo_banco', 'mesma_conta', 'mesma_moeda', 'pagamento_reinvestment',
       'source_account_id', 'target_account_id', 'transaction_id'],
      dtype='object')
Index(['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name'], dtype='object')


## 2. Preparação dos atributos das transações

A normalização é ajustada apenas no treino. Validação e teste usam o mesmo `scaler`, evitando vazamento de informação.

In [3]:
from sklearn.preprocessing import StandardScaler

cols = [
    'amount_paid_log',
    'amount_received_log',
    'razao_pago_recebido'
]

scaler = StandardScaler()

X_train_balanceado[cols] = scaler.fit_transform(X_train_balanceado[cols])
X_val_processado[cols] = scaler.transform(X_val_processado[cols])
X_test_processado[cols] = scaler.transform(X_test_processado[cols])

In [4]:
cat_cols = [
    "Receiving Currency",
    "Payment Currency",
    "Payment Format"
]

num_cols = [
    "amount_paid_log",
    "amount_received_log",
    "dif_valor",
    "dif_valor_abs",
    "razao_pago_recebido",
    "mesmo_banco",
    "mesma_conta",
    "mesma_moeda",
    "pagamento_reinvestment",
    "hora_sin",
    "hora_cos",
    "dia_semana_sin",
    "dia_semana_cos",
    "mes_sin",
    "mes_cos",
    "fim_de_semana",
    "inicio_mes",
    "fim_mes",
    "semana_mes"
]

X_train_edges = pd.get_dummies(
    X_train_balanceado[num_cols + cat_cols],
    columns=cat_cols,
    drop_first=False
)

X_val_edges = pd.get_dummies(
    X_val_processado[num_cols + cat_cols],
    columns=cat_cols,
    drop_first=False
)

X_test_edges = pd.get_dummies(
    X_test_processado[num_cols + cat_cols],
    columns=cat_cols,
    drop_first=False
)

# Validação e teste precisam ter exatamente as mesmas colunas do treino.
X_val_edges = X_val_edges.reindex(columns=X_train_edges.columns, fill_value=0)
X_test_edges = X_test_edges.reindex(columns=X_train_edges.columns, fill_value=0)

print(X_train_edges.shape, X_val_edges.shape, X_test_edges.shape)

(27450, 56) (5882, 56) (5883, 56)


In [5]:
def limpar_features_edges(X_edges):
    X_edges = X_edges.copy()

    # Converte booleanos e categorias codificadas para número.
    X_edges = X_edges.apply(pd.to_numeric, errors="coerce")

    # Remove NaN e infinitos.
    X_edges = X_edges.replace([np.inf, -np.inf], 0)
    X_edges = X_edges.fillna(0)

    # Garante tipo compatível com PyTorch.
    X_edges = X_edges.astype(np.float32)

    return X_edges

X_train_edges = limpar_features_edges(X_train_edges)
X_val_edges = limpar_features_edges(X_val_edges)
X_test_edges = limpar_features_edges(X_test_edges)

print(X_train_edges.dtypes.value_counts())

float32    56
Name: count, dtype: int64


# Arvore

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import pandas as pd
import numpy as np

In [7]:
X_train_tab = X_train_edges.copy()
X_val_tab = X_val_edges.copy()
X_test_tab = X_test_edges.copy()

y_train_tab = X_train_balanceado["Is Laundering"].astype(int)
y_val_tab = X_val_processado["Is Laundering"].astype(int)
y_test_tab = X_test_processado["Is Laundering"].astype(int)

print("Treino:", X_train_tab.shape, y_train_tab.shape)
print("Validação:", X_val_tab.shape, y_val_tab.shape)
print("Teste:", X_test_tab.shape, y_test_tab.shape)

Treino: (27450, 56) (27450,)
Validação: (5882, 56) (5882,)
Teste: (5883, 56) (5883,)


In [8]:
def avaliar_sklearn(modelo, X, y, tipo="Teste", threshold=0.5):
    y_prob = modelo.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y, y_prob)
    except ValueError:
        auc = 0

    try:
        auprc = average_precision_score(y, y_prob)
    except ValueError:
        auprc = 0

    metricas = {
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "auprc": auprc
    }

    print(
        f"{tipo}: "
        f"acc={acc:.3f} | precision={precision:.3f} | "
        f"recall={recall:.3f} | f1={f1:.3f} | "
        f"auc={auc:.3f} | auprc={auprc:.3f}"
    )

    return metricas, y_pred, y_prob

In [9]:
#Testar idiferentes combinações de parametros

parametros_arvore = []

for max_depth in [3, 5, 8, 10, 15, 20, None]:
    for min_samples_leaf in [1, 5, 10, 20, 50]:
        for class_weight in [None, "balanced", {0: 1, 1: 10}]:

            modelo = DecisionTreeClassifier(
                criterion="gini",
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                class_weight=class_weight,
                random_state=42
            )

            modelo.fit(X_train_tab, y_train_tab)

            metricas_val, _, _ = avaliar_sklearn(
                modelo,
                X_val_tab,
                y_val_tab,
                tipo="Validação",
                threshold=0.5
            )

            parametros_arvore.append({
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "class_weight": class_weight,
                **metricas_val
            })

            print("-" * 60)

resultados_arvore_val = pd.DataFrame(parametros_arvore)
display(resultados_arvore_val.sort_values("f1", ascending=False).head(10))

Validação: acc=0.920 | precision=0.555 | recall=0.598 | f1=0.576 | auc=0.855 | auprc=0.434
------------------------------------------------------------
Validação: acc=0.900 | precision=0.468 | recall=0.727 | f1=0.569 | auc=0.846 | auprc=0.371
------------------------------------------------------------
Validação: acc=0.900 | precision=0.468 | recall=0.727 | f1=0.569 | auc=0.846 | auprc=0.371
------------------------------------------------------------
Validação: acc=0.920 | precision=0.555 | recall=0.598 | f1=0.576 | auc=0.855 | auprc=0.434
------------------------------------------------------------
Validação: acc=0.900 | precision=0.468 | recall=0.727 | f1=0.569 | auc=0.846 | auprc=0.371
------------------------------------------------------------
Validação: acc=0.900 | precision=0.468 | recall=0.727 | f1=0.569 | auc=0.846 | auprc=0.371
------------------------------------------------------------
Validação: acc=0.920 | precision=0.555 | recall=0.598 | f1=0.576 | auc=0.855 | auprc=0.4

,max_depth,min_samples_leaf,class_weight,acc,precision,recall,f1,auc,auprc
30,8.0,1,None,0.928936,0.620123,0.564486,0.590998,0.863685,0.509983
42,8.0,50,None,0.927746,0.609127,0.573832,0.590953,0.872679,0.545492
48,10.0,5,None,0.926216,0.596190,0.585047,0.590566,0.849375,0.514723
51,10.0,10,None,0.926046,0.595785,0.581308,0.588458,0.865392,0.535987
45,10.0,1,None,0.925196,0.589118,0.586916,0.588015,0.852308,0.505087
33,8.0,5,None,0.927576,0.609658,0.566355,0.587209,0.863947,0.517294
36,8.0,10,None,0.927746,0.613636,0.555140,0.582924,0.866012,0.519016
39,8.0,20,None,0.927236,0.609407,0.557009,0.582031,0.867205,0.523645
21,5.0,10,None,0.925706,0.596457,0.566355,0.581016,0.867217,0.469668
27,5.0,50,None,0.925706,0.596457,0.566355,0.581016,0.867756,0.470552


In [11]:
# Selecionar a arvore com melhor f1
melhor_config_arvore = resultados_arvore_val.sort_values(
    "f1",
    ascending=False
).iloc[0]

melhor_config_arvore

,30
max_depth,8.0
min_samples_leaf,1
class_weight,None
acc,0.928936
precision,0.620123
recall,0.564486
f1,0.590998
auc,0.863685
auprc,0.509983


In [12]:
modelo_arvore = DecisionTreeClassifier(
    criterion="gini",
    max_depth=int(melhor_config_arvore["max_depth"]),
    min_samples_leaf=int(melhor_config_arvore["min_samples_leaf"]),
    class_weight=melhor_config_arvore["class_weight"],
    random_state=42
)

modelo_arvore.fit(X_train_tab, y_train_tab)

metricas_arvore, y_pred_arvore, y_prob_arvore = avaliar_sklearn(
    modelo_arvore,
    X_test_tab,
    y_test_tab,
    tipo="Teste - Árvore de Decisão",
    threshold=0.5
)

Teste - Árvore de Decisão: acc=0.928 | precision=0.616 | recall=0.546 | f1=0.579 | auc=0.847 | auprc=0.487


In [13]:
resultado_arvore_df = pd.DataFrame([
    {
        "modelo": "Árvore de Decisão - transações isoladas",
        "accuracy": metricas_arvore["acc"],
        "precision": metricas_arvore["precision"],
        "recall": metricas_arvore["recall"],
        "f1": metricas_arvore["f1"],
        "auc": metricas_arvore["auc"],
        "auprc": metricas_arvore["auprc"]
    }
])

display(resultado_arvore_df)

,modelo,accuracy,precision,recall,f1,auc,auprc
0,Árvore de Decisão - transações isoladas,0.927758,0.616034,0.545794,0.578791,0.847329,0.487079
